@@markdown
# derived_8.4-eval-mlp-1.3 — one more shot at the 2-regime MLP (1.75 H100-hours)

Follow-up to `derived_8.4-eval-mlp-1.2` (2-seed val-selected winners: 2regime_96 `w512x512x512_d0.3_lr1e-3` test R² 0.761, 2regime_54 `w512x512x512_d0.3_huber0.1` 0.765; test-best reference 0.789; XGBoost 2-regime 0.815). 1.2 documented that the remaining gap is (1) **systematic positive test bias** (bias² ≈ 10–17% of MSE for the 96-family) and (2) **test error bottoming out early** (~ep 90) while val stays flat, so patience-60 picks a late epoch. 1.3 gives the current architecture (plain 2-regime MLP) one final shot within a **1.75 H100-hour budget**, attacking those two inefficiencies plus completing 2-seeding of 1.2's strong 1-seed configs and testing train-time regularizers (EMA, mixup, target centering, stronger wd, batch 256). The two offline analyses of 1.2's saved artifacts (per-cluster calibration and early-stopping-rule replay) are reported first; both are documented negatives, so the sweep focuses on the train-time levers.


In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import json

# Robust resolution of the experiment dir whether executed from notebooks/ or in-place.
candidates = [Path.cwd() / "experiment/derived_8.4-eval-mlp-1.3", Path.cwd()]
EXP_DIR = next((p for p in candidates if (p / "metrics_summary.csv").exists()), Path.cwd())

df_summary = pd.read_csv(EXP_DIR / "metrics_summary.csv")
df_per_regime = pd.read_csv(EXP_DIR / "per_regime_metrics_summary.csv")
df_sweep = pd.read_csv(EXP_DIR / "sweep_results.csv")
df_ood = pd.read_csv(EXP_DIR / "ood_summary.csv")
df_timing = pd.read_csv(EXP_DIR / "timing_summary.csv")
df_cal12 = pd.read_csv(EXP_DIR / "calibration_12_summary.csv")
df_stop12 = pd.read_csv(EXP_DIR / "stopping_12_summary.csv")
df_stop12_agg = pd.read_csv(EXP_DIR / "stopping_12_aggregate.csv")
with open(EXP_DIR / "selected_features.json") as f:
    selected_meta = json.load(f)
with open(EXP_DIR / "timing_log.json") as f:
    timing_log = json.load(f)

fam_labels = {"2regime_96": "2-Regime-96", "2regime_54": "2-Regime-54"}
print("loaded", len(df_summary), "leaderboard rows,", len(df_sweep), "sweep rows,", len(df_cal12), "calibration rows (1.2)")

loaded 28 leaderboard rows, 38 sweep rows, 55 calibration rows (1.2)


## Selection Protocol v4 Diagnostic

1.2 found the val (2021–22) ranking barely transfers to the test period and that the aux2020 holdout (2020 ⊂ train) measures *train fit*, not generalization — so it was **dropped as a selection signal**. 1.3 selects configs by **2-seed mean val RMSE** (phase-2 2nd seed on the val top-10 MLP configs per family) and keeps aux2020 as a diagnostic only. This section reports the val ranking, the Spearman correlations vs test, and what each rule would have picked, so the selection is auditable (no test-based cherry-picking).

In [2]:
from scipy.stats import spearmanr
print("### Selection Protocol v4 Diagnostic (selection = 2-seed mean val RMSE)")
for family, fam_label in fam_labels.items():
    sub = df_sweep[df_sweep["family"] == family].dropna(subset=["test_r2"]).copy()
    if sub.empty:
        continue
    sub = sub.sort_values("val_rmse", na_position="last").reset_index(drop=True)
    print(f"\n#### {fam_label} — top-10 by val RMSE")
    cols = ["config_id", "architecture", "n_seeds", "val_rmse", "aux_rmse", "robust_score", "test_r2", "test_rmse"]
    show = sub.head(10)[cols].copy()
    robust_rank = sub.sort_values("robust_score")["config_id"].tolist()
    show["robust_rank"] = [robust_rank.index(c) + 1 for c in show["config_id"]]
    print(show.to_markdown(index=False))
    for metric, label in [("val_rmse", "val_rmse"), ("robust_score", "robust_score"), ("aux_rmse", "aux_rmse")]:
        valid = sub.dropna(subset=[metric, "test_r2"])
        if len(valid) >= 8:
            rho, p = spearmanr(valid[metric], valid["test_r2"])
            print(f"  Spearman({label}, test_r2) = {rho:+.3f} (p={p:.3f}, n={len(valid)})")
    mlp_sub = sub[sub["architecture"] == "mlp"]
    val_mlp = mlp_sub.sort_values("val_rmse").iloc[0]
    test_best = sub.sort_values("test_r2", ascending=False).iloc[0]
    print(f"  val winner (MLP) : {val_mlp['config_id']} (test_r2={val_mlp['test_r2']:.4f})")
    print(f"  test best (ref)  : {test_best['config_id']} (test_r2={test_best['test_r2']:.4f})")

### Selection Protocol v4 Diagnostic (selection = 2-seed mean val RMSE)

#### 2-Regime-96 — top-10 by val RMSE
| config_id                   | architecture   |   n_seeds |   val_rmse |   aux_rmse |   robust_score |   test_r2 |   test_rmse |   robust_rank |
|:----------------------------|:---------------|----------:|-----------:|-----------:|---------------:|----------:|------------:|--------------:|
| w512x512x512_d0.3_lr1e-3    | mlp            |         2 |  0.0482834 |  0.0249457 |      0.0366146 |  0.761018 |   0.0497987 |             3 |
| w512x512x512_d0.3_huber0.05 | mlp            |         2 |  0.0484297 |  0.0248293 |      0.0366295 |  0.770174 |   0.0488354 |             4 |
| w512x512x512_d0.3_huber0.02 | mlp            |         2 |  0.0486934 |  0.0226742 |      0.0356838 |  0.762261 |   0.049669  |             1 |
| w512x512x512_d0.3_huber0.1  | mlp            |         2 |  0.0491336 |  0.02649   |      0.0378118 |  0.761771 |   0.0497203 |             7 |
| w512x512x51

## Overall Model Leaderboard

All evaluated models ranked by pooled test R² over 2023–2025 (6,620 samples, 7 WA stations). MLP rows carry the sweep `config_id` and their `n_seeds`; `(val top-k avg)` rows are offline seed-averaged ensembles of the top-k val-selected MLP configs (no extra training); `(calibrated, ...)` rows use per-cluster affine calibration fit on val (added only if it beats raw — in practice it never does, see the Calibration section). XGBoost rows are the eval-1.1 references; `MLP-1.2` rows are the 1.2 val-selected winners + test-best references; `test-best` rows are reported for reference only (selection on test would be leakage).

In [3]:
cols = ["model_name", "strategy_name", "pooled_r2", "pooled_rmse", "pooled_ubrmse", "pooled_bias", "pooled_mae", "pooled_pearson"]
print("### Overall Leaderboard (2023-2025 Test Set)")
print(df_summary[cols].to_markdown(index=False))

### Overall Leaderboard (2023-2025 Test Set)
| model_name                                                | strategy_name          |   pooled_r2 |   pooled_rmse |   pooled_ubrmse |   pooled_bias |   pooled_mae |   pooled_pearson |
|:----------------------------------------------------------|:-----------------------|------------:|--------------:|----------------:|--------------:|-------------:|-----------------:|
| Clustering_V0_Full_k2 (Winner c0=0, c1=10)                | XGBoost_Reference      |    0.81496  |     0.0438196 |       0.043337  |   0.00648567  |    0.0337195 |         0.905594 |
| MLP 2-Regime-54 (test-best, w384x384_d0.3_gelu)           | MLP_testbest_reference |    0.788821 |     0.0468125 |       0.0467953 |   0.00126699  |    0.0362252 |         0.888558 |
| MLP-1.2 2-Regime-54 (test_best: w384x384_d0.3_gelu)       | MLP_1.2_Reference      |    0.788821 |     0.0468125 |       0.0467953 |   0.00126699  |    0.0362252 |         0.888558 |
| MLP 2-Regime-54 (test-best, 

## Hyperparameter Sweep Summary

34 curated configs (1.2 anchors re-run under the v5 protocol, 1.2's strong 1-seed configs for 2-seed completion, and new gap-targeting configs: EMA, mixup α=0.2, target centering, huber δ=0.02, wd 1e-2, batch 256, and the 54-family's good-capacity region) trained in the 2-regime families with 8 parallel H100 workers. Configs are ranked by **2-seed mean val RMSE** (the honest selection signal); test R² is reported for reference. Phase-2 configs carry `n_seeds=2`.

In [4]:
for family, fam_label in fam_labels.items():
    sub = df_sweep[df_sweep["family"] == family].sort_values("val_rmse", na_position="last").head(10)
    print(f"### Sweep Top-10 — {fam_label} (by val RMSE, the honest selection signal)")
    cols = ["config_id", "n_seeds", "dropout", "lr", "loss", "val_rmse", "aux_rmse", "robust_score", "test_r2", "test_rmse", "best_epoch", "train_time_s"]
    show = sub[cols].copy()
    if "ema" not in show.columns and "ema" in df_sweep.columns:
        pass
    print(show.to_markdown(index=False))
    print()

### Sweep Top-10 — 2-Regime-96 (by val RMSE, the honest selection signal)
| config_id                   |   n_seeds |   dropout |     lr | loss   |   val_rmse |   aux_rmse |   robust_score |   test_r2 |   test_rmse |   best_epoch |   train_time_s |
|:----------------------------|----------:|----------:|-------:|:-------|-----------:|-----------:|---------------:|----------:|------------:|-------------:|---------------:|
| w512x512x512_d0.3_lr1e-3    |         2 |       0.3 | 0.001  | mse    |  0.0482834 |  0.0249457 |      0.0366146 |  0.761018 |   0.0497987 |          263 |        95.435  |
| w512x512x512_d0.3_huber0.05 |         2 |       0.3 | 0.0003 | huber  |  0.0484297 |  0.0248293 |      0.0366295 |  0.770174 |   0.0488354 |          260 |        97.8043 |
| w512x512x512_d0.3_huber0.02 |         2 |       0.3 | 0.0003 | huber  |  0.0486934 |  0.0226742 |      0.0356838 |  0.762261 |   0.049669  |          220 |        86.4382 |
| w512x512x512_d0.3_huber0.1  |         2 |       0

## Per-Regime Performance Breakdown

Cluster 0 holds 73% of the test rows, so it dominates the pooled R². The 2-regime specialists' per-cluster test metrics are shown for the val top-3 MLP configs per family, the XGBoost references, and the 1.2 reference winners.

In [5]:
print("### Per-Regime Performance Breakdown")
cols = ["strategy_name", "model_name", "cluster", "n_train", "n_test", "r2", "rmse", "ubrmse", "bias", "mae"]
print(df_per_regime[cols].to_markdown(index=False))

### Per-Regime Performance Breakdown
| strategy_name     | model_name                                    |   cluster |   n_train |   n_test |       r2 |      rmse |    ubrmse |        bias |       mae |
|:------------------|:----------------------------------------------|----------:|----------:|---------:|---------:|----------:|----------:|------------:|----------:|
| MLP_2regime_96    | MLP 2-Regime-96 (w512x512x512_d0.3_lr1e-3)    |         0 |      7156 |     4817 | 0.754287 | 0.0495899 | 0.0472413 | 0.0150802   | 0.0389389 |
| MLP_2regime_96    | MLP 2-Regime-96 (w512x512x512_d0.3_lr1e-3)    |         1 |      2647 |     1803 | 0.776352 | 0.0503523 | 0.0440792 | 0.0243389   | 0.0370033 |
| MLP_2regime_96    | MLP 2-Regime-96 (w512x512x512_d0.3_huber0.05) |         0 |      7156 |     4817 | 0.770502 | 0.0479257 | 0.0459775 | 0.0135256   | 0.0370535 |
| MLP_2regime_96    | MLP 2-Regime-96 (w512x512x512_d0.3_huber0.05) |         1 |      2647 |     1803 | 0.768879 | 0.0511867 | 0.043

## Yearly Performance Breakdown

Year-by-year R² on the 2023–2025 test period — 1.0's analysis showed MLPs degrade most in 2025 (distribution drift); this table tracks whether the 1.3 regularizers close that year specifically.

In [6]:
year_cols = [c for c in df_summary.columns if c.startswith("year_") and c.endswith("_r2")]
print("### Year-by-Year R² Breakdown")
print(df_summary[["model_name", "pooled_r2", *year_cols]].to_markdown(index=False))

### Year-by-Year R² Breakdown
| model_name                                                |   pooled_r2 |   year_2023_r2 |   year_2024_r2 |   year_2025_r2 |
|:----------------------------------------------------------|------------:|---------------:|---------------:|---------------:|
| Clustering_V0_Full_k2 (Winner c0=0, c1=10)                |    0.81496  |       0.822971 |       0.783256 |       0.83029  |
| MLP 2-Regime-54 (test-best, w384x384_d0.3_gelu)           |    0.788821 |       0.773579 |       0.818284 |       0.770357 |
| MLP-1.2 2-Regime-54 (test_best: w384x384_d0.3_gelu)       |    0.788821 |       0.773579 |       0.818284 |       0.770357 |
| MLP 2-Regime-54 (test-best, w384x384_d0.3)                |    0.78409  |       0.755911 |       0.818958 |       0.775181 |
| MLP-1.2 2-Regime-96 (test_best: w512x512x512_d0.4)        |    0.783883 |       0.747432 |       0.826116 |       0.777354 |
| MLP 2-Regime-96 (test-best, w256x256_d0.5)                |    0.783404 |      

## Offline Analysis of 1.2 — Per-Cluster Calibration (documented negative)

1.2 found the 2-regime MLPs carry a systematic positive test bias (bias² ≈ 10–17% of MSE). The obvious fix — fit a per-cluster (and global) affine map `y' = a·y + b` on the **val** predictions of every saved 1.2 model and apply it to test — **does not transfer**: calibrated test R² is *worse* for nearly every config and the test bias grows. The val→test bias relationship is period-specific (consistent with 1.2's "val is a weak proxy for test"). Raw predictions stand; `run_mlp_eval.py` only reports calibrated rows when they beat raw (in practice none do).

In [7]:
print("### Per-cluster affine calibration on 1.2 models (fit on val -> test)")
print("Raw vs calibrated pooled test R² / bias. A calibration that helps would show cal_pc_r2 > raw_r2.")
for family, fam_label in fam_labels.items():
    sub = df_cal12[df_cal12["family"] == family].sort_values("raw_r2", ascending=False)
    print(f"\n#### {fam_label} — top-8 by raw R²")
    cols = ["config_id", "n_seeds", "raw_r2", "cal_pc_r2", "cal_g_r2", "raw_rmse", "cal_pc_rmse", "raw_bias", "cal_pc_bias"]
    print(sub.head(8)[cols].to_markdown(index=False))
    helped = (sub["cal_pc_r2"] > sub["raw_r2"]).sum()
    print(f"configs where per-cluster calibration HELPED on test: {helped}/{len(sub)}")
    print(f"median raw_r2 {sub['raw_r2'].median():.4f} -> cal_pc_r2 {sub['cal_pc_r2'].median():.4f}")

### Per-cluster affine calibration on 1.2 models (fit on val -> test)
Raw vs calibrated pooled test R² / bias. A calibration that helps would show cal_pc_r2 > raw_r2.

#### 2-Regime-96 — top-8 by raw R²
| config_id                   |   n_seeds |   raw_r2 |   cal_pc_r2 |   cal_g_r2 |   raw_rmse |   cal_pc_rmse |   raw_bias |   cal_pc_bias |
|:----------------------------|----------:|---------:|------------:|-----------:|-----------:|--------------:|-----------:|--------------:|
| w512x512x512_d0.4           |         1 | 0.783883 |    0.758638 |   0.75857  |  0.0473565 |     0.0500461 | 0.0144227  |     0.0222451 |
| w256x256_d0.5               |         1 | 0.783404 |    0.727816 |   0.718496 |  0.047409  |     0.0531456 | 0.00493258 |     0.0264161 |
| w512x512x512_d0.3_gelu      |         2 | 0.777923 |    0.757836 |   0.758278 |  0.0480052 |     0.0501292 | 0.0159755  |     0.0217491 |
| w1024x1024x512_d0.3_gelu    |         2 | 0.774762 |    0.761033 |   0.76234  |  0.0483455 |   

## Offline Analysis of 1.2 — Early-Stopping-Rule Replay (patience-60 confirmed)

1.2's winner's test error bottomed out at ~ep 90 while val stayed flat to ep 260. We replayed alternative **honest** epoch-selection rules on every saved 1.2 per-epoch curve (val/aux/test): patience-60 (baseline), val+aux joint minimum, and first-sustained-plateau variants. **Finding: no honest rule beats patience-60** — the plateau rules stop too early (undertrained), and val+aux helps the 54-family slightly but hurts the 96-family. The oracle (argmin test) bound shows ~0.004 RMSE of headroom that is not honestly reachable. The 1.3 trainer therefore keeps patience-60.

In [8]:
print("### Early-stopping-rule replay on 1.2 curves (mean pooled test RMSE over configs/seeds)")
print("Lower is better. 'oracle' = argmin on test (reference bound, not achievable honestly).")
for family, fam_label in fam_labels.items():
    sub = df_stop12_agg[df_stop12_agg["family"] == family].sort_values("mean_test_rmse")
    print(f"\n#### {fam_label}")
    print(sub[["rule", "mean_test_rmse", "median_test_rmse", "n"]].to_markdown(index=False))

### Early-stopping-rule replay on 1.2 curves (mean pooled test RMSE over configs/seeds)
Lower is better. 'oracle' = argmin on test (reference bound, not achievable honestly).

#### 2-Regime-96
| rule             |   mean_test_rmse |   median_test_rmse |   n |
|:-----------------|-----------------:|-------------------:|----:|
| oracle           |        0.0476222 |          0.0477377 |  69 |
| patience60       |        0.0520159 |          0.0519324 |  69 |
| patience40       |        0.0520159 |          0.0519324 |  69 |
| patience20       |        0.0520159 |          0.0519324 |  69 |
| val_aux          |        0.0537652 |          0.0528206 |  69 |
| plateau_w60e1e-4 |        0.0555998 |          0.0547605 |  69 |
| plateau_w40e3e-4 |        0.060673  |          0.0598781 |  69 |
| plateau_w40e1e-4 |        0.060673  |          0.0598781 |  69 |
| plateau_w20e1e-4 |        0.0674415 |          0.0677908 |  69 |

#### 2-Regime-54
| rule             |   mean_test_rmse |   median_tes

## New Trainer Knobs (1.3) — EMA / mixup / target-centering / wd / bs

The sweep's new configs target the two documented inefficiencies at train time: EMA (weight averaging), mixup (α=0.2), target centering (per-specialist residual learning), stronger weight decay (1e-2), and batch 256 (more gradient noise). This table shows their val/test results alongside their non-knobbed anchors.

In [9]:
knob_configs = [c for c in df_sweep["config_id"].unique()
                if any(k in c for k in ("_ema", "_mixup", "_centered", "_wd1e-2", "_bs256", "_huber0.02"))]
sub = df_sweep[df_sweep["config_id"].isin(knob_configs)].sort_values(["family", "val_rmse"])
cols = ["family", "config_id", "n_seeds", "val_rmse", "aux_rmse", "test_r2", "test_rmse", "test_bias", "best_epoch"]
print("### New-knob configs (val top + test reference)")
print(sub[cols].to_markdown(index=False))

### New-knob configs (val top + test reference)
| family     | config_id                       |   n_seeds |   val_rmse |   aux_rmse |     test_r2 |   test_rmse |   test_bias |   best_epoch |
|:-----------|:--------------------------------|----------:|-----------:|-----------:|------------:|------------:|------------:|-------------:|
| 2regime_54 | w384x384_d0.3_huber0.02         |         2 |  0.0574505 |  0.0247853 |    0.766799 |   0.0491928 |  0.0153431  |          240 |
| 2regime_54 | w384x384_d0.3_mixup0.2          |         2 |  0.06208   |  0.035957  |    0.778403 |   0.0479532 |  0.00454811 |          301 |
| 2regime_54 | w384x384_d0.3_gelu_ema          |         1 |  0.184839  |  0.240567  | -163.444    |   1.3063    |  1.27094    |            2 |
| 2regime_54 | w512x512x512_d0.3_huber0.1_ema  |         2 |  0.197799  |  0.190174  |  -45.173    |   0.692197  |  0.664229   |           20 |
| 2regime_96 | w512x512x512_d0.3_huber0.02     |         2 |  0.0486934 |  0.0226742 |  

## Extrapolation (OOD) Check

Test rows whose top-10 gain features fall outside the trainval [min, max] range are flagged as OOD (same definition as mlp-1.1/1.2) — 588/6,620 rows (8.9%). Compares the best neural models per family vs the XGBoost references on in-distribution vs OOD slices.

In [10]:
print("### OOD Slice Metrics (best neural model per family vs XGBoost references)")
print(df_ood[["model", "slice", "n", "r2", "rmse", "bias", "mae"]].to_markdown(index=False))

### OOD Slice Metrics (best neural model per family vs XGBoost references)
| model                                       | slice           |    n |       r2 |      rmse |        bias |       mae |
|:--------------------------------------------|:----------------|-----:|---------:|----------:|------------:|----------:|
| MLP 2regime-96 (w512x512x512_d0.3_lr1e-3)   | all             | 6620 | 0.761018 | 0.0497987 |  0.0176019  | 0.0384117 |
| MLP 2regime-96 (w512x512x512_d0.3_lr1e-3)   | in_distribution | 6032 | 0.755427 | 0.0512552 |  0.0204626  | 0.0397959 |
| MLP 2regime-96 (w512x512x512_d0.3_lr1e-3)   | ood             |  588 | 0.750658 | 0.0311459 | -0.0117442  | 0.0242127 |
| MLP 2regime-54 (w512x512x512_d0.3_huber0.1) | all             | 6620 | 0.76511  | 0.0493706 |  0.00681535 | 0.0385003 |
| MLP 2regime-54 (w512x512x512_d0.3_huber0.1) | in_distribution | 6032 | 0.766711 | 0.0500588 |  0.00927805 | 0.0388175 |
| MLP 2regime-54 (w512x512x512_d0.3_huber0.1) | ood             |  588 

## Overfitting-Symptom Analysis

Quantifies the generalization failure modes of the 1.3 sweep from the saved artifacts (no retraining): (1) the **train-fit vs held-out gap** — aux2020 RMSE vs val vs test; (2) **capacity vs transfer** by n_params bucket; (3) the **per-epoch curve shape** of each family's 2-seed val winner — does test still bottom out early and then rise?; (4) **systematic bias** on test vs XGBoost. Backed by `analyze_overfitting.py`.

In [11]:
import sys
sys.path.insert(0, str(EXP_DIR))
from analyze_overfitting import compute_overfitting, print_report
_overfit = compute_overfitting(df_sweep, EXP_DIR)
print_report(_overfit)

OVERFITTING-SYMPTOM ANALYSIS — derived_8.4-eval-mlp-1.3 (from sweep artifacts)

### 1. Train-fit vs held-out gap (median RMSE over 2-regime MLP configs)
| family     |   aux2020 (train-fit) |   val |   test |   val/train ratio |
|:-----------|----------------------:|------:|-------:|------------------:|
| 2regime_96 | 0.0295 | 0.0505 | 0.0497 | 1.7x |
| 2regime_54 | 0.0293 | 0.0599 | 0.0487 | 2.0x |

### 2. Capacity vs test transfer (median by n_params bucket)
| family     | capacity   |   n_configs |   med_val_rmse |   med_test_r2 |   med_test_bias |
|:-----------|:-----------|------------:|---------------:|--------------:|----------------:|
| 2regime_96 | <200k | 1 | 0.0569 | 0.7834 | 0.0049 |
| 2regime_96 | 200-500k | 1 | 0.0509 | 0.7698 | 0.0146 |
| 2regime_96 | 500k-1M | 5 | 0.0506 | 0.7541 | 0.0185 |
| 2regime_96 | 1M+ | 14 | 0.0503 | 0.7617 | 0.0203 |
| 2regime_54 | <200k | 1 | 0.0605 | 0.7741 | 0.0060 |
| 2regime_54 | 200-500k | 7 | 0.0614 | 0.7784 | 0.0056 |
| 2regime_54 | 500

## Timing

Sweep wall time, total GPU-seconds vs the 1.75 H100-hour budget, and per-config training times.

In [12]:
print("### Timing (H100 PCIe 80 GB, 8 parallel workers)")
print(f"Total sweep wall time (final invocation): {timing_log.get('sweep_wall_s', float('nan')):.1f} s  |  eval wall time: {timing_log.get('eval_wall_s', float('nan')):.1f} s")
jobs = timing_log.get("jobs", {})
total_gpu_s = sum(j.get("train_time_s", 0.0) for j in jobs.values())
print(f"Total training time (all jobs, GPU-seconds): {total_gpu_s:.0f} s = {total_gpu_s/3600:.2f} GPU-hours (budget: 1.75)")
print("\n### Per-Config Timing (top-8 fastest/slowest by train time)")
tfast = df_timing.sort_values("train_time_s").head(8)
tslow = df_timing.sort_values("train_time_s").tail(8)
cols = ["family", "config_id", "n_seeds", "train_time_s", "epochs", "best_epoch", "val_rmse", "test_r2"]
print("fastest:\n", tfast[cols].to_markdown(index=False))
print("\nslowest:\n", tslow[cols].to_markdown(index=False))

### Timing (H100 PCIe 80 GB, 8 parallel workers)
Total sweep wall time (final invocation): 162.3 s  |  eval wall time: 5.8 s
Total training time (all jobs, GPU-seconds): 3144 s = 0.87 GPU-hours (budget: 1.75)

### Per-Config Timing (top-8 fastest/slowest by train time)
fastest:
 | family     | config_id                       |   n_seeds |   train_time_s |   epochs |   best_epoch |   val_rmse |     test_r2 |
|:-----------|:--------------------------------|----------:|---------------:|---------:|-------------:|-----------:|------------:|
| 2regime_54 | w384x384_d0.3_gelu_ema          |         1 |        10.7431 |       62 |            2 |  0.184839  | -163.444    |
| 2regime_96 | w1024x512x256_d0.3_gelu_ema     |         1 |        12.3074 |       63 |            3 |  0.168236  |  -55.8832   |
| 2regime_96 | w512x512x512_d0.3_lr1e-3_ema    |         1 |        14.7494 |       81 |           21 |  0.160122  |  -23.1512   |
| 2regime_96 | w512x512x512_d0.3_huber0.05_ema |         1 |     

@@markdown
## Key Takeaways

1. **The val-selected winners are bit-identical to 1.2** (deterministic seeds): 2regime_96 `w512x512x512_d0.3_lr1e-3` 0.7610, 2regime_54 `w512x512x512_d0.3_huber0.1` 0.7651 — the anchors reproduce 1.2 exactly under the v5 protocol. The gap to the XGBoost 2-regime winner (0.815) is unchanged (~0.03), so the **plain-2-regime-MLP ceiling is confirmed**.
2. **The two offline fixes were refuted before spending GPU.** Per-cluster affine calibration fit on val does not transfer to test (calibrated R² is worse for 54-family 0/12 configs and 43/48 of 96-family; medians 0.774→0.746 / 0.754→0.735). No honest early-stopping rule beats patience-60 (plateau rules stop too early; val+aux helps 54 slightly but hurts 96). Both are documented negatives.
3. **EMA is a documented trainer-level failure**: the inherited decay-0.999-per-step EMA lags ~70 epochs behind the fast-moving head layer given ~14 steps/epoch, so EMA-evaluated val never falls below ~0.15 and test R² is catastrophic (−20 to −160). Excluded from selection; root cause isolated (the EMA head alone causes the blowup).
4. **What did move the needle (modestly):** the 54-family's good-capacity region — `w448x448_d0.3_gelu` (2-seed test R² 0.7809, val rank 11, near-zero bias +0.001) and `w384x384x256_d0.3_gelu` (0.7695, val rank 3) beat the 54 val winner on test; the 54-family **val top-10 avg 0.7825** edges past the XGBoost global baseline (0.779). mixup helped the 54 family (`w384x384_d0.3_mixup0.2` 0.7784, 2-seed); target centering (0.732) and batch 256 (0.692) hurt.
5. **Extrapolation advantage holds:** the 96-family winner remains the best OOD model (OOD R² 0.751 vs XGBoost 2-regime 0.619).
6. **Budget:** 0.87 of the 1.75 H100-hours used (3,144 GPU-seconds; sweep wall 162 s at 8 workers).


## Data Split and Protocol

All models use the official `derived_8.4` split (`data/splits/derived_8.4/`, see `split_meta.json`): a **temporal** split over the 7 WA stations (BeaverPass_WA_990, CayusePass_WA, Darrington, Paradise_WA, Quinault, SourdoughGulch_WA_985, Spokane), derived from `derived_8.3` with `MartenRidge_WA_999` and `RainyPass_WA_711` filtered out. train = 2017–2020 (n=9,803), val = 2021–2022 (n=4,805), test = 2023–2025 (n=6,620).

- **train (2017–2020)** — the only data ever trained on. Preprocessing follows the repo LSTM convention (`_clean_inf` → median imputation → standardization → clip [−5, 5]) and is fit **on train only** (per-cluster subset for the 2-regime specialists); the target stays in original units (RMSE-consistent with the XGBoost baselines).
- **val (2021–2022)** — the selection oracle. Training early-stops with patience 60 on val RMSE, the best-val-epoch checkpoint (`best_model.pt`) produces the test predictions, and configs are selected by **2-seed mean val RMSE** (phase 2 adds seed 7 to the val top-10 MLP configs per family, expanded to top-15 in the final run). Test is never used for any of these choices.
- **test (2023–2025)** — untouched until the end; scored exactly once at the best-val epoch. `test-best` rows in the leaderboard are reported **for reference only** (selection on test would be leakage).
- **aux2020 diagnostic (not a selection signal):** the 2020 slice of train (n=2,519) is preprocessed with the same train-fitted imputer/scaler and scored at the best-val epoch. Because 2020 ⊂ train, aux2020 RMSE measures *train fit*, not generalization — 1.3 therefore dropped it from selection (1.2's robust score used it; documented as a failed signal).
- **Cluster assignment (2-regime families):** the `Clustering_V0_Full_k2` router (KMeans k=2 on the 50 V0 features, seed 42) is fit on **trainval** (train + val), then predicts labels for train/val/test — the test split never influences cluster memberships. Each specialist trains on its cluster's slice of train; pooled val RMSE = √(Σ (n_c/n_total) · rmse_c²).


## Reproducibility Notes

- **Protocol (data_version 5):** train on train (2017–2020, n=9,803), early-stop / select configs on the official val split (2021–2022, n=4,805), evaluate on the untouched test set (2023–2025, n=6,620). Final winners selected by **2-seed mean val RMSE** among plain MLPs. No trainval retrain (documented negative in 1.2).
- **Preprocessing:** median imputation + standardization fit on train only, clip to [−5, 5]; target in original units.
- **Training:** AdamW + warmup (5%) + cosine LR, grad clip 1.0, patience 60; 2-seed sweep (seeds {42, 7}); new knobs (ema, mixup_alpha, center_target, stop_rule) default to 1.2 behavior.
- **Offline analyses:** `analyze_calibration.py` (per-cluster affine fit on val) and `analyze_stopping.py` (rule replay on saved curves) run on the 1.2 artifacts and again on 1.3's — zero GPU, fully reproducible from saved checkpoints/curves.
- **Reproduce:** `uv run --no-sync python run_mlp_sweep.py --resume` → `uv run --no-sync python run_mlp_eval.py` → `uv run --no-sync python analyze_calibration.py --exp-dir ../derived_8.4-eval-mlp-1.2 --tag 12` (and `--tag 13`) → `analyze_stopping.py` (tags 12/13) → `analyze_overfitting.py` → `analyze_extrapolation.py` → `nb execute derived_8.4-eval-mlp-1.3.ipynb` (from `notebooks/`).